# Day 2 - Topic 1: Functions Basics

> Lead-Level Data Science Interview Prep Series

## 1. Introduction

- A **function** is a named, reusable block of code that performs one specific task
- You define it once, then call it as many times as needed
- Why needed?
  - Avoids repeating the same code (DRY principle - Don't Repeat Yourself)
  - Breaks big problems into small, testable pieces
  - Makes code readable: `clean_data(df)` explains itself
- Where used?
  - Every real Data Science codebase is organized into functions (load_data, clean_data, train_model, evaluate)
  - Pandas `.apply()` takes your custom functions
  - Writing clean functions is a direct interview evaluation criterion at Lead level

## 2. Real-Life Analogy

- A function is like a **coffee machine**:
  - Inputs (parameters): coffee beans, water, milk
  - Process (function body): the machine's internal steps
  - Output (return value): a cup of coffee
- You don't rebuild the machine every morning - you just press the button (call the function) with your chosen inputs
- Default arguments are like the machine's default setting: "medium sugar unless you specify otherwise"
- `*args` is like a juicer that accepts ANY number of fruits; `**kwargs` is like a pizza order form with any number of labeled toppings ("cheese=extra", "olives=no")

## 3. Explanation

- Define a function with `def`, a name, parameters in `()`, and an indented body
- `return` sends a value back to the caller and ends the function; without `return`, a function returns `None`
- **Parameter types (in order):**
  1. Positional: matched by position - `greet("Asha")`
  2. Keyword: matched by name - `greet(name="Asha")`
  3. Default: has a fallback value - `def greet(name="Guest")`
  4. `*args`: collects extra positional arguments into a **tuple**
  5. `**kwargs`: collects extra keyword arguments into a **dict**
- Parameter order rule in a definition: `def f(positional, default=x, *args, **kwargs)`

> **Trick to remember:** `*args` = "star grabs a tuple of leftovers", `**kwargs` = "double star grabs a dict of labeled leftovers". Position first, names later.

## 4. Syntax

```python
def function_name(param1, param2=default_value, *args, **kwargs):
    """Docstring: what this function does."""
    # body
    return result
```

- `def` - keyword that starts a function definition
- `function_name` - snake_case by convention
- `param2=default_value` - default parameter (optional for caller)
- `*args` - tuple of extra positional arguments
- `**kwargs` - dict of extra keyword arguments
- `return` - sends back the result (optional)
- Docstring - triple-quoted description right under `def`, readable via `help(function_name)`

In [ ]:
def greet(name, greeting="Hello"):
    """Return a greeting message for the given name."""
    return f"{greeting}, {name}!"

print(greet("Asha"))                     # uses default greeting
print(greet("Vikram", greeting="Hi"))    # overrides default


## 5. Examples

### Basic Example

In [ ]:
# Basic: a function with a return value
def add(a, b):
    return a + b

result = add(10, 5)
print(result)

# A function without return gives None
def say_hello():
    print("Hello")

x = say_hello()
print(x)   # None


### Intermediate Example

In [ ]:
# Intermediate: *args and **kwargs
def order_summary(customer, *items, **details):
    print(f"Customer: {customer}")
    print(f"Items (tuple): {items}")
    print(f"Details (dict): {details}")

order_summary("Neha", "pizza", "cola", table=4, takeaway=False)


- `customer` grabs the first positional argument
- `*items` collects the remaining positional arguments `("pizza", "cola")` into a tuple
- `**details` collects the keyword arguments `{"table": 4, "takeaway": False}` into a dict
- This pattern lets one function accept flexible inputs - heavily used inside libraries like Pandas/Matplotlib (that is why their functions accept dozens of optional keyword arguments)

### Real-World Example

In [ ]:
# Real-world: a reusable data-validation function for EDA
def validate_numeric_column(values, min_val=0, max_val=1_000_000):
    """Split raw values into clean numbers and rejected entries."""
    clean, rejected = [], []
    for v in values:
        if isinstance(v, (int, float)) and min_val <= v <= max_val:
            clean.append(v)
        else:
            rejected.append(v)
    return clean, rejected     # returning multiple values (as a tuple)

salaries = [52000, -100, 68000, "NA", 49000, 99999999]
clean_salaries, bad_salaries = validate_numeric_column(salaries, min_val=10000, max_val=500000)

print("Clean:", clean_salaries)
print("Rejected:", bad_salaries)


- One function now validates ANY numeric column with configurable bounds - reusable across every dataset you touch
- `return clean, rejected` returns two values at once - Python packs them into a tuple, and `clean_salaries, bad_salaries = ...` unpacks them (tuple unpacking)
- `isinstance(v, (int, float))` checks against multiple types in one call
- This "return multiple values + unpack" pattern is everywhere in real DS code (e.g. `X_train, X_test, y_train, y_test = train_test_split(...)`)

## 6. Internal Working

- When Python reads `def`, it creates a **function object** in memory and binds the name to it - the body does NOT run yet
- The body runs only when the function is **called**
- Each call creates a new **local scope** (namespace) - variables inside are destroyed when the call ends
- Name lookup follows the **LEGB rule**: Local -> Enclosing -> Global -> Built-in
- Functions are **first-class objects**: they can be stored in variables, passed as arguments, and returned from other functions (this powers map/filter/decorators later)

> **Trick to remember:** LEGB = "Little Elephants Grow Big" - Python searches for a variable name from the innermost scope outward.

In [ ]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print(x)      # finds Local first (LEGB)
    inner()

outer()

# Functions are objects - can be assigned and passed around
def square(n):
    return n * n

f = square          # no () - assigning the function itself, not calling it
print(f(6))


## 7. Time and Space Complexity

- Defining a function: O(1)
- Calling a function: O(1) overhead for the call itself (creating the local scope/frame) - the body's complexity depends entirely on what the body does
- Each active call consumes stack space - relevant later for recursion (deep recursion can hit Python's recursion limit, ~1000 by default)

> **Interview note:** Function call overhead is why a plain Python loop calling a function per row is slower than vectorized NumPy - millions of tiny call overheads add up. This connects directly to Day 3.

## 8. Common Mistakes

- Using a **mutable default argument** (`def f(items=[])`) - the SAME list is shared across all calls (the classic Python interview trap, shown below)
- Forgetting `return` and wondering why the result is `None`
- Calling with wrong argument order when mixing positional and keyword args (positional must always come first in a call)
- Shadowing built-ins by naming a function/variable `list`, `sum`, `type`, etc.
- Writing giant functions that do many things - hard to test, hard to reuse (one function = one job)

In [ ]:
# THE classic trap: mutable default argument
def add_item_bad(item, items=[]):
    items.append(item)
    return items

print(add_item_bad("a"))   # ['a']
print(add_item_bad("b"))   # ['a', 'b']  <- surprise! same list reused across calls

# Correct pattern: use None as the default
def add_item_good(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

print(add_item_good("a"))  # ['a']
print(add_item_good("b"))  # ['b']  <- fresh list each call


- Why it happens: default values are evaluated **once, at definition time**, not at every call - so the same list object lives on between calls
- Fix: default to `None`, create the list inside the body
- This is one of the most frequently asked Python interview questions at every level

## 9. Best Practices

- One function = one clear job; if the name needs "and" in it, split it
- Use snake_case names that describe the action: `clean_column_names`, not `ccn`
- Always write a short docstring for non-trivial functions
- Use `None` (never `[]` or `{}`) as the default for mutable parameters
- Prefer returning values over printing inside functions - lets callers decide what to do with results
- Use keyword arguments at call sites for clarity: `validate(values, min_val=0)` reads better than `validate(values, 0)`

## 10. Interview Questions

**Beginner**
- Q: What does a function return if there is no `return` statement?
  A: `None` - every Python function returns something, and the default is `None`.
- Q: What is the difference between a parameter and an argument?
  A: A parameter is the variable name in the function definition; an argument is the actual value passed when calling the function.

**Intermediate**
- Q: What do `*args` and `**kwargs` do?
  A: `*args` collects extra positional arguments into a tuple; `**kwargs` collects extra keyword arguments into a dict. They let a function accept a flexible number of inputs.
- Q: Why is `def f(items=[])` dangerous?
  A: Default values are evaluated once at definition time, so the same list object is shared across all calls - items accumulate unexpectedly. Use `items=None` and create the list inside.

**Advanced**
- Q: What does it mean that Python functions are first-class objects?
  A: Functions can be assigned to variables, stored in data structures, passed as arguments, and returned from other functions - enabling patterns like callbacks, `map`/`filter`, and decorators.
- Q: Explain the LEGB rule.
  A: When resolving a variable name, Python searches scopes in order: Local (inside the current function), Enclosing (any outer function), Global (module level), Built-in (Python's own names). The first match wins.

## 11. Practice Problems

**Easy**
1. Write a function `is_even(n)` that returns `True`/`False`.
2. Write a function with a default argument that greets a user, defaulting to "Guest" when no name is given.

**Medium**
3. Write a function `stats(*numbers)` that returns the min, max, and average of any number of arguments as a tuple, then unpack the result into three variables.
4. Write a function `build_profile(name, **details)` that returns a dict containing the name plus all extra keyword details passed in.

**Hard**
5. Demonstrate the mutable default argument bug with your own example, then fix it with the `None` pattern, and add a comment explaining exactly WHEN Python evaluates default values.

## 12. Revision Summary

- `def` creates a function object; body runs only when called
- No `return` means the function returns `None`
- Parameter order: positional, default, `*args`, `**kwargs`
- `*args` -> tuple of extra positional args; `**kwargs` -> dict of extra keyword args
- Returning multiple values = returning a tuple; unpack with `a, b = f()`
- Mutable default argument (`items=[]`) is the classic trap - use `None` instead
- Scope lookup follows LEGB: Local, Enclosing, Global, Built-in
- Functions are first-class objects - foundation for lambda/map/filter/decorators (next topics)

> **Next topic (Day 2 continues):** Lambda, map, filter, reduce